# Fixed-time vs. joint transition-time likelihoods

This analysis compares runs using the known transition times (`inference_first_N_trees.out`) with runs that jointly estimate transition times and the other parameters (`inference_joint_times_first_N_trees.out`), for the same setting, batch, and tree count. It reads the final `LL=` value from each output file.

The paired difference is **fixed-time LL − joint-time LL**. A positive value means the fixed-time run achieved a higher likelihood; a negative value means the joint-time run did.

In [1]:
from pathlib import Path
import math
import re
import statistics

candidate_roots = (
    Path("data/simulation/coord_diff/star"),
    Path("../data/simulation/coord_diff/star"),
)
DATA_DIR = next(
    (path.resolve() for path in candidate_roots if path.is_dir()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find data/simulation/coord_diff/star from the current directory"
    )

LL_PATTERN = re.compile(r"^LL=([^\s]+)\s*$", re.MULTILINE)

def read_log_likelihood(path):
    matches = LL_PATTERN.findall(path.read_text())
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one LL= line in {path}; got {len(matches)}")
    value = float(matches[0])
    if not math.isfinite(value):
        raise ValueError(f"Non-finite likelihood in {path}: {value}")
    return value

def collect_pairs(num_trees):
    fixed_name = f"inference_first_{num_trees}_trees.out"
    joint_name = f"inference_joint_times_first_{num_trees}_trees.out"
    fixed_paths = {
        (path.parent.parent.name, path.parent.name): path
        for path in DATA_DIR.glob(f"setting_*/batch_*/{fixed_name}")
    }
    joint_paths = {
        (path.parent.parent.name, path.parent.name): path
        for path in DATA_DIR.glob(f"setting_*/batch_*/{joint_name}")
    }
    pairs = []
    for setting, batch in sorted(fixed_paths.keys() & joint_paths.keys()):
        fixed_ll = read_log_likelihood(fixed_paths[(setting, batch)])
        joint_ll = read_log_likelihood(joint_paths[(setting, batch)])
        pairs.append({
            "num_trees": num_trees,
            "setting": setting,
            "batch": batch,
            "fixed_ll": fixed_ll,
            "joint_ll": joint_ll,
            "fixed_minus_joint": fixed_ll - joint_ll,
        })
    return len(fixed_paths), len(joint_paths), pairs

results = {num_trees: collect_pairs(num_trees) for num_trees in (4, 8)}

In [2]:
print(f"Data: {DATA_DIR}")
print("Higher likelihood means a better fit. Positive delta = fixed-time LL > joint-time LL.")
print()
print(f"{'trees':>5} {'fixed files':>12} {'joint files':>12} {'paired':>7} "
      f"{'fixed higher':>13} {'joint higher':>13} {'ties':>5} "
      f"{'median delta':>13} {'min delta':>10} {'max delta':>11}")

TIE_TOLERANCE = 1e-8
for num_trees, (fixed_count, joint_count, pairs) in results.items():
    deltas = [pair["fixed_minus_joint"] for pair in pairs]
    fixed_higher = sum(delta > TIE_TOLERANCE for delta in deltas)
    joint_higher = sum(delta < -TIE_TOLERANCE for delta in deltas)
    ties = len(deltas) - fixed_higher - joint_higher
    print(f"{num_trees:5d} {fixed_count:12d} {joint_count:12d} {len(pairs):7d} "
          f"{fixed_higher:13d} {joint_higher:13d} {ties:5d} "
          f"{statistics.median(deltas):13.6f} {min(deltas):10.6f} {max(deltas):11.6f}")

Data: /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/star
Higher likelihood means a better fit. Positive delta = fixed-time LL > joint-time LL.

trees  fixed files  joint files  paired  fixed higher  joint higher  ties  median delta  min delta  max delta
    4          101           71       9             6             3     0      0.214748  -3.945738  716.761814
    8          118           87       9             8             1     0      6.251860  -1.206340  166.953832


In [3]:
print(f"{'trees':>5} {'setting':<11} {'batch':<8} {'fixed-time LL':>17} "
      f"{'joint-time LL':>17} {'fixed − joint':>16}")
for num_trees, (_, _, pairs) in results.items():
    for pair in pairs:
        print(f"{num_trees:5d} {pair['setting']:<11} {pair['batch']:<8} "
              f"{pair['fixed_ll']:17.6f} {pair['joint_ll']:17.6f} "
              f"{pair['fixed_minus_joint']:16.6f}")

trees  setting     batch       fixed-time LL       joint-time LL       fixed − joint
    4  setting_19  batch_5       431.713880       435.659618        -3.945738
    4  setting_2   batch_0     12760.912830     12745.370800        15.542030
    4  setting_34  batch_6         8.255036         8.216287         0.038749
    4  setting_43  batch_3      2201.102147      2202.525109        -1.422963
    4  setting_57  batch_0      1702.480120      1704.884957        -2.404837
    4  setting_60  batch_2     23579.714073     23579.321203         0.392870
    4  setting_72  batch_5      5868.760448      5845.158460        23.601988
    4  setting_89  batch_9     11451.827937     10735.066123       716.761814
    4  setting_99  batch_2       199.179463       198.964714         0.214748
    8  setting_48  batch_8      1075.207454      1076.413794        -1.206340
    8  setting_55  batch_5      2512.775382      2508.134409         4.640973
    8  setting_60  batch_5     15494.779637     15480.592

## Interpretation

Among the **9 matched runs per tree count**, the fixed-time run has the higher likelihood in **6/9 (4 trees)** and **8/9 (8 trees)**. Across both groups, fixed-time likelihood is higher in **14 of 18** matched runs. This supports the concern that some current joint-time runs have not reached a solution as good as the known-time fit; the large positive differences in a few runs are especially notable.

The comparison is paired on setting and batch, and both rules use the same tree count and likelihood normalization. A joint optimization can represent the fixed-time solution by choosing those same times, so its optimum should be at least as high as the fixed-time result. These are optimization runs rather than proofs of global optima, however: a positive difference shows the **reported joint result** is worse than that feasible fixed-time result, but does not isolate whether the missed improvement is due to the transition-time search, the rate/probability optimization, or both.

Coverage is currently limited: the data directory contains 101 fixed-time and 71 joint-time outputs for 4 trees, and 118 fixed-time and 87 joint-time outputs for 8 trees, but only 9 same-setting/same-batch pairs at each size. The statistics above use only those matched pairs. Re-run the code cells after more joint outputs complete to refresh the comparison.